# Once Upon a Mirror, notebook 4: Epsilon-greedy

In section 4 of the game, the mirror spun a roulette every morning after the initialization: on the large slice,
🥇, it showed the product with the highest estimate, and on the small slice, 🎩, one spin in five, it drew a card from the
hat, any product at random. Here you write that mirror in Python, in the game's four steps: initialize, spin,
sample, update. Then you play it on the game's race season against the Queen's mirror and the mirrors of
notebooks 2 and 3.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one only
if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 25
minutes.

The products, the customers and their probabilities are invented for the game. Start with the three folded cells
below: the mirror's kit, the checks, and the mirrors of notebooks 2 and 3.

In [ ]:
#@title The mirror's kit (run me first) { display-mode: "form" }
# The mirror's kit. Run this cell first; you do not need to read it.
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

def estimate(row):
    """The estimate of a product: the average of its 1s and 0s (taps / showings); None when it was never shown."""
    if len(row) == 0:
        return None
    return sum(row) / len(row)


def estimates(rows):
    """The estimate of every product on the board, in the order of the rows."""
    return [estimate(row) for row in rows]

def pick_highest(values, next_number):
    """The row with the highest value. When several rows share the highest value, one of them at random, with a
    number from next_number: the rule of the game, so the same stream gives the same choice. A number is drawn only
    when there is a tie."""
    best = max(values)
    tied = [k for k in range(len(values)) if abs(values[k] - best) <= 1e-9]
    if len(tied) == 1:
        return tied[0]
    return tied[int(next_number() * len(tied))]

In [ ]:
#@title The checks of notebook 4 (run me too) { display-mode: "form" }
# The checks of notebook 4, play and show_mornings. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

# Task A's tests, one morning each from a season of the game: "board" is the board before that morning and "skip"
# the numbers that the mirror drew on the earlier mornings. "kind" is what the game's roulette does: "g" 🥇 the
# highest estimate, "t" 🥇 with a tie broken at random, "h" 🎩 a card from the hat. "product" is the row it shows,
# "nums" the numbers it draws that morning, "ahead" the next three numbers of the stream from that morning on, and
# "tied" the rows that share the highest estimate.
_N4_MORNINGS = [{'about': "the race season at Cinderella's", 'board': [[0], [0], [0], [1], [0]], 'seed': 103993, 'skip': 0, 'eps': 0.2, 'morning': 5, 'kind': 'g', 'product': 3, 'nums': [0.31993313832208514], 'ahead': [0.31993313832208514, 0.5980246248655021, 0.0729405942838639], 'tied': [3], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "the race season at Cinderella's", 'board': [[0], [0], [0], [1, 0, 0], [0]], 'seed': 103993, 'skip': 2, 'eps': 0.2, 'morning': 7, 'kind': 'h', 'product': 4, 'nums': [0.0729405942838639, 0.8934970367699862], 'ahead': [0.0729405942838639, 0.8934970367699862, 0.8264451266732067], 'tied': [], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "the race season at Cinderella's", 'board': [[0], [0], [0], [1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], [0, 0]], 'seed': 103993, 'skip': 13, 'eps': 0.2, 'morning': 17, 'kind': 'h', 'product': 0, 'nums': [0.08165588323026896, 0.17405192810110748], 'ahead': [0.08165588323026896, 0.17405192810110748, 0.8301818557083607], 'tied': [], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "the race season at Cinderella's", 'board': [[0, 1], [0], [0], [1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], [0, 0]], 'seed': 103993, 'skip': 15, 'eps': 0.2, 'morning': 18, 'kind': 'g', 'product': 0, 'nums': [0.8301818557083607], 'ahead': [0.8301818557083607, 0.5351533985231072, 0.7369110982399434], 'tied': [0], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "another season at Rapunzel's", 'board': [[0], [0], [0], [0], [0]], 'seed': 10, 'skip': 0, 'eps': 0.2, 'morning': 5, 'kind': 't', 'product': 4, 'nums': [0.28640562761574984, 0.9466124968603253], 'ahead': [0.28640562761574984, 0.9466124968603253, 0.13344237278215587], 'tied': [0, 1, 2, 3, 4], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "another season at Rapunzel's", 'board': [[1, 0], [1, 1, 1, 1], [0], [0], [0]], 'seed': 7, 'skip': 7, 'eps': 0.2, 'morning': 9, 'kind': 'h', 'product': 4, 'nums': [0.12113402830436826, 0.9530127921607345], 'ahead': [0.12113402830436826, 0.9530127921607345, 0.30959586217068136], 'tied': [], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}, {'about': "another season at Rapunzel's", 'board': [[1, 0, 1, 0], [0], [1, 0, 0], [0], [0]], 'seed': 9, 'skip': 7, 'eps': 0.2, 'morning': 10, 'kind': 'g', 'product': 0, 'nums': [0.986585165373981], 'ahead': [0.986585165373981, 0.8935359490569681, 0.07190737267956138], 'tied': [0], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav']}]

# Task B's tests, one season each, and what the game's epsilon-greedy does on it, morning by morning: "shown" holds
# the row shown, "dots" her answer (1 a tap, 0 no tap), "kinds" the step ("i" initialize, then "g", "t" or "h" as
# above) and "ties" the rows that share the highest estimate on each morning with a tie.
_N4_SEASONS = [{'about': "the race season at Cinderella's", 'call': 'epsilon_greedy(cinderella, season_moods(RACE), strategy_numbers(RACE, "epsilon-greedy"), eps)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'eps': 0.2, 'shown': '0123433433333333300000300000000000000000000010000140112000000010000000000000000000304030000040003000', 'dots': '0001000000001100111111011001111101001110000001010001000011110000000111111001111111110100110001100000', 'kinds': 'iiiiigghggggggggghgggghggggggggggggggggggggghgggghhghhhggggggghggggggggggggggggggghhhghggggghggghggg', 'ties': {}}, {'about': "the race season at Rapunzel's", 'call': 'epsilon_greedy(rapunzel, season_moods(RACE), strategy_numbers(RACE, "epsilon-greedy"), eps)', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'eps': 0.2, 'shown': '0123434111111111011113111111111111111111111111111411121111111111111111111111111113041311111411131111', 'dots': '0101000111111101111110111101111101101111001011110011001011111001010111111001111110101001111011100010', 'kinds': 'iiiiithggggggggghgggghggggggggggggggggggggghgggghhghhhggggggghggggggggggggggggggghhhghggggghggghgggg', 'ties': {5: [1, 3]}}, {'about': "another season at Cinderella's", 'call': 'epsilon_greedy(cinderella, season_moods(1), strategy_numbers(1, "epsilon-greedy"), eps)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 1, 'mornings': 100, 'eps': 0.2, 'shown': '0123411001000002000300030100000000020000000000001313000000000000000000000040000120030000000000000040', 'dots': '0100000010111111111011101011010101001011111011110000011000000011010100100101111001110011101101111100', 'kinds': 'iiiiigghhhggggghggghggghghhgggggggghhghgghghgggghhhhgggggggggggggggggggghghgggghhgghggggghgggggggghg', 'ties': {}}, {'about': 'the collection Ready for the ball, from section 3', 'call': 'epsilon_greedy([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(2), strategy_numbers(2, "epsilon-greedy"), eps)', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 2, 'mornings': 100, 'eps': 0.2, 'shown': '0123401131142121133221111111111111111131121111011211011141110111333313111333303333333333330321433331', 'dots': '0111001111111111110101111011011111110110101111001101001111111100111011110111101111111111111110111111', 'kinds': 'iiiiihttttthttttthhttggggggggggggggggghgghgggghgghgghggghggghgggggggghtgggggghgggggggggggghghhhggggh', 'ties': {6: [1, 2, 3], 7: [1, 2, 3], 8: [1, 2, 3], 9: [1, 2, 3], 10: [1, 2, 3], 12: [1, 2, 3], 13: [1, 2, 3], 14: [1, 2, 3], 15: [1, 2, 3], 16: [1, 2, 3], 19: [1, 2], 20: [1, 2], 70: [1, 3]}}, {'about': "another season at Rapunzel's", 'call': 'epsilon_greedy(rapunzel, season_moods(3), strategy_numbers(3, "epsilon-greedy"), eps)', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 3, 'mornings': 100, 'eps': 0.2, 'shown': '0123431211111111111101111111111111111112111411111113111111111121110311111111121111121211111111111101', 'dots': '0101001111111110111100110101111111110110111010111110111110011001110011111111001111101000010001101111', 'kinds': 'iiiiitghgggggggggggghgggggggggggggggggghggghgggggghhgggggggggghggghhggggggggghggggghghgggggggggggghg', 'ties': {5: [1, 3]}}, {'about': "another season at Cinderella's", 'call': 'epsilon_greedy(cinderella, season_moods(4), strategy_numbers(4, "epsilon-greedy"), eps)', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 4, 'mornings': 100, 'eps': 0.2, 'shown': '0123433303031332333333433333333334443333333313333434332333330333333333333133303331333333313333233330', 'dots': '0001001000010000000000001001010001001010101000110010010000010100110001000010010010100011110100011001', 'kinds': 'iiiiiggghghghgghgggggghgggggggggghgtgggggggghggghhghgghggggghgggggggggggghggghghghggggggghgggghggggh', 'ties': {35: [3, 4]}}, {'about': "a short season of 12 mornings at Rapunzel's", 'call': 'epsilon_greedy(rapunzel, season_moods(5)[:12], strategy_numbers(5, "epsilon-greedy"), eps)', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 5, 'mornings': 12, 'eps': 0.2, 'shown': '012344423111', 'dots': '010010000111', 'kinds': 'iiiiihhhhggg', 'ties': {}}]
_N4_SUCCESS = {'spin_the_roulette': "spin_the_roulette works on 7 mornings, number for number as in the game: 🥇 the highest estimate, 🎩 a card from the hat, and a tie broken at random. On the 8th morning of Cinderella's race season the needle stops at 0.07, on 🎩, and the card is the satnav, as in the game.", 'epsilon_greedy': "epsilon_greedy works on 7 seasons, morning by morning as in the game. On Cinderella's race season the needle stops on 🎩 16 times in 95 spins, and the mirror gets 47 taps in 100 mornings, a rate of 0.47: the game's numbers."}
_N4_TASKS = {"spin_the_roulette": "task A", "epsilon_greedy": "task B"}
_N4_NEEDS = {"spin_the_roulette": [], "epsilon_greedy": ["spin_the_roulette"]}
_N4_DRAWS = {"i": 0, "g": 1, "t": 2, "h": 2}
_N4_GAPS = {}
_N4_WORKING = {}
_N4_LOG = []


def _n4_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n4_count(n, word):
    """1 dot, 3 dots"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _n4_ordinal(n):
    """1st, 2nd, 3rd, 4th, 11th, 21st"""
    if n % 100 in (11, 12, 13):
        return "%dth" % n
    return "%d%s" % (n, {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th"))


def _n4_morning(t):
    """morning 12 (the 13th of the season): Python counts the mornings from 0."""
    return "morning %d (the %s of the season)" % (t, _n4_ordinal(t + 1))


def _n4_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _n4_dec(x):
    """At most two decimals, as in the game: 0.6, 0.33, 1."""
    text = "%.2f" % x
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    return text


def _n4_spin(x, eps):
    """A spin with two decimals, or three when two would hide on which side of eps it stops."""
    text = "%.2f" % x
    if float(text) == float(eps):
        text = "%.3f" % x
    return text


def _n4_frac(row):
    """1/2 = 0.5"""
    return "%d/%d = %s" % (sum(row), len(row), _n4_dec(sum(row) / len(row)))


def _n4_rows_text(rows):
    """[[0, 1], [0, 0]]"""
    parts = []
    for row in rows:
        dots = []
        for x in row:
            dots.append(str(x))
        parts.append("[" + ", ".join(dots) + "]")
    return "[" + ", ".join(parts) + "]"


def _n4_product(test, k):
    """the rope ladder (row 1), with p when the test has it: the rope ladder (row 1, p = 0.2)"""
    if not isinstance(k, int) or isinstance(k, bool) or k < 0 or k >= len(test["names"]):
        return "row %s" % _n4_short(k)
    if "p" in test:
        return "the %s (row %d, p = %s)" % (test["names"][k], k, test["p"][k])
    return "the %s (row %d)" % (test["names"][k], k)


def _n4_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N4_GAPS:
        return _N4_GAPS[code]
    # A gap such as "spin = ..." stays in the compiled function; a gap on a line of its own, or in "if ...:", is only
    # visible in the source, which Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N4_GAPS[code] = gap
    return gap


def _n4_source(fn):
    """The lines of fn without their comments, or [] when the source cannot be read."""
    import inspect
    try:
        lines = []
        for line in inspect.getsource(fn).splitlines():
            lines.append(line.split("#")[0])
        return lines
    except Exception:
        return []


def _n4_is_board(value, rows=None):
    """True when value is a board: a list of lists of 1s and 0s (and rows lists, when rows is given)."""
    if not isinstance(value, list) or len(value) == 0:
        return False
    if rows is not None and len(value) != rows:
        return False
    for row in value:
        if not isinstance(row, list):
            return False
        for x in row:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return False
    return True


def _n4_names(fn):
    """The global names that the code of fn uses."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return ()
    return code.co_names


def _n4_uses(fn, other):
    """True when the code of fn calls the function named other."""
    return other in _n4_names(fn)


def _n4_python_random(fn):
    """True when fn draws from Python's own random numbers."""
    for word in ("random", "randint", "uniform", "choice", "randrange"):
        if word in _n4_names(fn):
            return True
    return False


def _n4_brackets_in_pick(fn):
    """True when fn hands pick_highest a number, next_number(), in the place of the stream itself."""
    for line in _n4_source(fn):
        if "pick_highest(" in line and "next_number()" in line.split("pick_highest(", 1)[1]:
            return True
    return False


# The check follows your mirror number by number. It hands your function a stream whose numbers know their place
# (the first number drawn is number 0), an eps that writes down each comparison, and moods and probabilities that
# write down each answer of the customer, which is what tap does. So it knows which numbers your mirror drew, which
# one it compared with eps and which product it showed, whatever the shape of your code.

def _n4_note_tap(prob, mood, swapped):
    """One comparison of a mood: ("tap", row, morning, swapped, value), row None when value is not one of p."""
    if isinstance(mood, _n4_Mood):
        row = None
        if isinstance(prob, _n4_Prob):
            row = prob.row
        _N4_LOG.append(("tap", row, mood.morning, swapped, prob))
    elif isinstance(prob, _n4_Prob) and isinstance(mood, _n4_Prob):
        _N4_LOG.append(("p", None, None, False, None))


def _n4_note_cmp(number, other, below):
    """One comparison of a number with other: ("cmp", its place in the stream or None, other, below, None). below is
    True when the comparison asks whether the number is below other, as spin < eps does."""
    place = None
    if isinstance(number, _n4_Number):
        place = number.place
    _N4_LOG.append(("cmp", place, other, below, None))


class _n4_Mood(float):
    """A mood that knows its morning: mood < p is her answer, the right way round."""

    def __lt__(self, other):
        _n4_note_tap(other, self, False)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n4_note_tap(other, self, False)
        return float.__le__(self, other)

    def __gt__(self, other):
        _n4_note_tap(other, self, True)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n4_note_tap(other, self, True)
        return float.__ge__(self, other)


class _n4_Prob(float):
    """A probability that knows its row: p > mood is her answer, the right way round."""

    def __gt__(self, other):
        _n4_note_tap(self, other, False)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n4_note_tap(self, other, False)
        return float.__ge__(self, other)

    def __lt__(self, other):
        _n4_note_tap(self, other, True)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n4_note_tap(self, other, True)
        return float.__le__(self, other)


class _n4_Number(float):
    """A number from the stream that knows its place: spin < eps asks whether it is below eps."""

    def __lt__(self, other):
        _n4_note_cmp(self, other, True)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n4_note_cmp(self, other, True)
        return float.__le__(self, other)

    def __gt__(self, other):
        _n4_note_cmp(self, other, False)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n4_note_cmp(self, other, False)
        return float.__ge__(self, other)


class _n4_Eps(float):
    """eps, which writes down each comparison: eps > spin asks whether the spin is below eps."""

    def __lt__(self, other):
        _n4_note_cmp(other, self, False)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n4_note_cmp(other, self, False)
        return float.__le__(self, other)

    def __gt__(self, other):
        _n4_note_cmp(other, self, True)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n4_note_cmp(other, self, True)
        return float.__ge__(self, other)


def _n4_stream(numbers, skip):
    """A next_number that gives the numbers of the stream numbers after the first skip, each one knowing its place,
    and writes each one down."""
    for j in range(skip):
        numbers()
    count = [0]

    def next_number():
        x = _n4_Number(numbers())
        x.place = count[0]
        count[0] = count[0] + 1
        _N4_LOG.append(("draw", x.place, None, False, float(x)))
        return x
    return next_number


def _n4_season_inputs(test):
    """Fresh inputs for a season test, [p, moods, next_number, eps], that write down what the mirror does."""
    p = []
    for k in range(len(test["p"])):
        x = _n4_Prob(test["p"][k])
        x.row = k
        p.append(x)
    moods = []
    plain = season_moods(test["seed"])[:test["mornings"]]
    for t in range(len(plain)):
        x = _n4_Mood(plain[t])
        x.morning = t
        moods.append(x)
    return [p, moods, _n4_stream(strategy_numbers(test["seed"], "epsilon-greedy"), 0), _n4_Eps(test["eps"])]


def _n4_run(fn, inputs):
    """Run fn on the inputs with its prints silenced: answer, error (None when the call went through), line (the line
    of your code where it happened) and log, what it did in order. Never raises."""
    import contextlib, io, traceback
    del _N4_LOG[:]
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*inputs)
    except Exception as error:
        # The check's numbers, eps, moods and probabilities are floats that write things down: name them float
        text = "%s: %s" % (type(error).__name__, error)
        for name in ("_n4_Number", "_n4_Eps", "_n4_Mood", "_n4_Prob"):
            text = text.replace(name, "float")
        run["error"] = text
        try:
            skip = (_n4_run.__code__.co_filename, tap.__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    run["log"] = list(_N4_LOG)
    del _N4_LOG[:]
    return run


def _n4_mornings_of(log):
    """The mornings of a run, one per answer of the customer: row, morning, swapped, value, and the places of the
    numbers drawn and the comparisons made since the previous answer. Also: p_read, True when two probabilities
    were compared, and eps_read, True when eps was compared with something."""
    mornings, draws, cmps = [], [], []
    p_read = False
    eps_read = False
    for kind, a, b, c, d in log:
        if kind == "tap":
            mornings.append({"row": a, "morning": b, "swapped": c, "value": d, "draws": draws, "cmps": cmps})
            draws, cmps = [], []
        elif kind == "draw":
            draws.append(a)
        elif kind == "cmp":
            cmps.append((a, b, c))
            if isinstance(b, _n4_Eps):
                eps_read = True
        else:
            p_read = True
    return mornings, p_read, eps_read


# The hints

def _n4_says(test, game):
    """What the game's roulette does on a morning, in words."""
    nums, product = game["nums"], game["product"]
    spin = _n4_spin(nums[0], test["eps"])
    shown = _n4_product(test, product)
    if game["kind"] == "h":
        return "the needle stops at %s, below eps, on 🎩, and the next number gives the card: %s" % (spin, shown)
    row = game["board"][product]
    if game["kind"] == "t":
        tied = []
        for k in game["tied"]:
            tied.append("the " + test["names"][k])
        return ("the needle stops at %s, on 🥇, and %s share the highest estimate, %s: the next number breaks the tie, %s"
                % (spin, _n4_and(tied), _n4_dec(sum(row) / len(row)), shown))
    return "the needle stops at %s, on 🥇: the highest estimate is %s, %s" % (spin, shown, _n4_frac(row))


def _n4_roulette_hint(name, fn, test, game, got_row, got_draws, cmps):
    """The hint for a morning on which the mirror drew other numbers or showed another product than the game's.
    cmps: (place in the morning, other, below) for each comparison with a number of the stream or with eps."""
    kind, product, ahead = game["kind"], game["product"], game["ahead"]
    k = len(game["board"])
    eps_places = []
    typed = False
    for place, other, below in cmps:
        if isinstance(other, _n4_Eps):
            eps_places.append(place)
        elif place is not None and isinstance(other, (int, float)) and not isinstance(other, bool):
            typed = True
    spin_line = "spin = next_number()"
    if name == "epsilon_greedy":
        spin_line = "product = spin_the_roulette(rows, next_number, eps)"
    if _n4_python_random(fn):
        return ("use the stream that the function gets, next_number: Python's own random numbers give another season "
                "than the game's. The spin is spin = next_number().")
    if _n4_uses(fn, "strategy_numbers") or _n4_uses(fn, "random_numbers"):
        return ("use the stream that the function gets, next_number: a new stream starts again from its first number "
                "on every call.")
    if _n4_brackets_in_pick(fn):
        return ("give pick_highest the stream itself, with no brackets after next_number: pick_highest(estimates(rows), "
                "next_number). With brackets, Python draws a number before the call, on every 🥇 morning.")
    if got_draws == 0:
        if name == "epsilon_greedy":
            return ("each morning after the initialization, spin the roulette first: " + spin_line + ", your function "
                    "from task A, draws the numbers from next_number.")
        return "spin the roulette first: spin = next_number() gives where the needle stops, a number between 0 and 1."
    if len(eps_places) == 0 and typed:
        if name == "epsilon_greedy" and _n4_uses(fn, "spin_the_roulette"):
            return "hand the roulette the eps that epsilon_greedy gets: spin_the_roulette(rows, next_number, eps)."
        return "compare the spin with eps, the size of the small slice that the function gets: if spin < eps."
    if name == "epsilon_greedy" and _n4_uses(fn, "spin_the_roulette"):
        # The roulette passed its check, so the difference comes from epsilon_greedy itself
        if got_draws != len(game["nums"]):
            return ("call spin_the_roulette once each morning after the initialization, and draw nothing else from "
                    "next_number: the roulette draws every number of the morning itself.")
        if got_row != product:
            return ("show the product that the roulette gives: product = spin_the_roulette(rows, next_number, eps), "
                    "with rows, the board so far.")
    if len(eps_places) > 0 and eps_places[0] is not None and eps_places[0] > 0:
        return ("the spin is the first number of the morning: spin = next_number() comes first, then if spin < eps. "
                "The hat's card is a new number, drawn after the spin and only when the needle stops on 🎩.")
    if kind == "h":
        if got_draws == 1 and got_row == int(ahead[0] * k):
            return ("the card is a new number from the stream: product = int(next_number() * len(rows)). The spin only "
                    "decides the slice, and a spin below eps would always give one of the top rows.")
        if got_draws == 1:
            return ("the needle stops on 🎩 when the spin is below eps: if spin < eps, a card from the hat, "
                    "int(next_number() * len(rows)). Your condition sends this spin to 🥇.")
        if got_draws == 2 and got_row != product:
            if got_row == 0:
                return ("int(next_number()) is 0 for any number below 1: multiply first, then take int: "
                        "int(next_number() * len(rows)).")
            if got_row == int(ahead[1] * (k - 1)):
                return ("multiply by len(rows), the number of products: int(next_number() * len(rows)) gives every row "
                        "from 0 to %d." % (k - 1))
            return ("the card is int(next_number() * len(rows)), from the second number of the morning, drawn right "
                    "after the spin.")
        if got_draws > 2:
            return "on 🎩 the mirror draws two numbers, the spin and then the card, and nothing else that morning."
    else:
        if got_draws >= 2 and got_row != product and got_row == int(ahead[1] * k):
            return ("the needle stops on 🎩 only when the spin is below eps, " + 'one spin in five' + ": if spin < eps, a card "
                    "from the hat; otherwise 🥇, the highest estimate. Your condition sends this spin to the hat.")
        if got_draws > len(game["nums"]) and got_row == product:
            return ("draw the hat's card only when the needle stops on 🎩: product = int(next_number() * len(rows)) "
                    "goes inside the if. A number drawn on a 🥇 morning shifts every spin after it.")
        if kind == "t" and (got_row in game["tied"] or got_draws < len(game["nums"])):
            return ("a tie is broken at random with a number from next_number, as in the game: "
                    "pick_highest(estimates(rows), next_number) does exactly that.")
        if got_row != product:
            return "🥇 is the row with the highest estimate on the board: pick_highest(estimates(rows), next_number)."
    return ("one number for the spin, spin = next_number(); on 🎩 one more for the card, int(next_number() * "
            "len(rows)); on 🥇 pick_highest(estimates(rows), next_number), which draws a number only for a tie.")


def _n4_explain_error(name, run, k):
    """The hint for a function that stopped with an error."""
    error = run["error"]
    text = (run["line"] or "").replace(" ", "")
    if "positional argument" in error:
        first = {"spin_the_roulette": "def spin_the_roulette(rows, next_number, eps):",
                 "epsilon_greedy": "def epsilon_greedy(p, moods, next_number, eps):"}[name]
        return "keep the first line of the function as it was given: " + first
    if error.startswith("NameError"):
        if "spin_the_roulette" in error:
            return "spin_the_roulette is your function from task A: run its cell, then this one."
        if "'p'" in error and name == "spin_the_roulette":
            return ("the roulette sees only the board: estimates(rows) gives the estimate of every row. Only tap reads p, "
                    "so it is out of the roulette's reach.")
        for word in ("random", "randint", "uniform", "choice"):
            if "'" + word + "'" in error:
                return "use the stream that the function gets: next_number() gives the next number, as in the game."
        known = "rows, next_number, eps, spin, product, estimates and pick_highest"
        if name == "epsilon_greedy":
            known = "p, moods, next_number, eps, rows, morning, product, dot, tap and spin_the_roulette"
        return "Python does not know a name in your function: check its spelling. Inside it you can use " + known + "."
    if error.startswith("TypeError"):
        if "NoneType" in error and "not supported" in error:
            if name == "epsilon_greedy":
                return ("estimates gives None for a product with no dots yet, and pick_highest cannot compare None. "
                        "Initialize first: on mornings 0 to len(p) - 1 show each product once, product = morning, and "
                        "spin the roulette from morning len(p) on.")
            return "spin is a number from the stream: spin = next_number(), then if spin < eps."
        if "'function'" in error and "subscriptable" in error:
            return "a function is called with round brackets: estimates(rows), next_number()."
        if "'function'" in error:
            return ("next_number() with brackets gives the next number: spin = next_number(). Without brackets it is "
                    "the stream itself.")
        if "not callable" in error and "float" in error:
            return ("give pick_highest the stream itself, with no brackets after next_number: "
                    "pick_highest(estimates(rows), next_number).")
        if name == "epsilon_greedy" and "has no len()" in error:
            return ("the roulette reads the board: spin_the_roulette(rows, next_number, eps), with rows, the dots so "
                    "far. Only tap reads p.")
        if "pick_highest(" in text and "list" in error:
            return ("🥇 is the row with the highest estimate on the board: pick_highest(estimates(rows), next_number). "
                    "pick_highest takes the list of estimates, one number per row.")
        if "can't multiply sequence" in error:
            return "multiply by len(rows), the number of products: int(next_number() * len(rows))."
        if "indices must be integers" in error and "float" in error:
            return ("a row number is a whole number: int(next_number() * len(rows)) turns a number from the stream "
                    "into one.")
        if "not supported between" in error:
            if name == "epsilon_greedy":
                return ("tap compares two numbers: the probability of the product shown and the mood of the morning, "
                        "tap(p[product], moods[morning]).")
            return "compare the spin, a number, with eps: if spin < eps."
    if error.startswith("IndexError"):
        if "moods[" in text:
            return ("a morning went past the end of the season: the remaining mornings are range(len(p), len(moods)), up "
                    "to the last morning of the season.")
        return ("a row goes from 0 to %d: int(next_number() * len(rows)) and pick_highest(estimates(rows), next_number) "
                "give exactly that." % (k - 1))
    if error.startswith("AttributeError") and "append" in error:
        return ("append works on a list. rows[product] is the row of the product shown, a list, and "
                "rows[product].append(dot) adds the dot at its end.")
    if name == "spin_the_roulette":
        return ("spin = next_number(); if spin < eps, a card from the hat, int(next_number() * len(rows)); otherwise "
                "pick_highest(estimates(rows), next_number).")
    return ("initialize, then on each remaining morning: product = spin_the_roulette(rows, next_number, eps), dot = "
            "tap(p[product], moods[morning]), rows[product].append(dot).")


def _n4_roulette_failure(fn):
    """Task A: (head, what, hint) for the first morning that fn gets wrong, or None when it gets them all right."""
    eps_read = False
    for test in _N4_MORNINGS:
        board = []
        for row in test["board"]:
            board.append(list(row))
        numbers = _n4_stream(strategy_numbers(test["seed"], "epsilon-greedy"), test["skip"])
        run = _n4_run(fn, [board, numbers, _n4_Eps(test["eps"])])
        draws, cmps = 0, []
        for kind, a, b, c, d in run["log"]:
            if kind == "draw":
                draws = draws + 1
            elif kind == "cmp":
                cmps.append((a, b, c))
                if isinstance(b, _n4_Eps):
                    eps_read = True
        answer = run["answer"]
        right = (run["error"] is None and isinstance(answer, int) and not isinstance(answer, bool)
                 and answer == test["product"] and draws == len(test["nums"]))
        if right:
            continue
        head = ("spin_the_roulette on the %s morning of %s, with rows = %s"
                % (_n4_ordinal(test["morning"] + 1), test["about"], _n4_rows_text(test["board"])))
        if run["error"] is not None:
            if "ellipsis" in run["error"].lower():
                return "gap"
            where = ""
            if run["line"]:
                where = " on the line  %s" % run["line"]
            return (head, "your function stopped with an error%s. %s" % (where, run["error"]),
                    _n4_explain_error("spin_the_roulette", run, len(test["board"])))
        if answer is Ellipsis:
            return "gap"
        if answer is None:
            return (head, "your function gave no answer (None)",
                    "end the function with return product, after the if and the else.")
        if not isinstance(answer, int) or isinstance(answer, bool):
            if isinstance(answer, _n4_Number) and answer.place == 0:
                return (head, "your function gave %s, the spin" % _n4_short(answer),
                        "the spin only decides the slice. Return the row of the product to show: return product.")
            if isinstance(answer, float):
                return (head, "your function gave %s, and a row is a whole number from 0 to %d"
                        % (_n4_short(answer), len(test["board"]) - 1),
                        "int drops the decimals: int(next_number() * len(rows)) gives a row.")
            return (head, "your function gave %s" % _n4_short(answer),
                    "the roulette returns one row, the product to show this morning: return product.")
        game = {"kind": test["kind"], "product": test["product"], "nums": test["nums"], "ahead": test["ahead"],
                "tied": test["tied"], "board": test["board"]}
        what = "%s. Your function gave %s and drew %s from next_number" % (
            _n4_says(test, game), _n4_product(test, answer), _n4_count(draws, "number"))
        return (head, what, _n4_roulette_hint("spin_the_roulette", fn, test, game, answer, draws, cmps))
    if not eps_read:
        return ("spin_the_roulette on the %d mornings of its check" % len(_N4_MORNINGS),
                "your roulette makes the game's choices, and it never compares the spin with eps",
                "use the eps that the function gets, the size of the small slice: if spin < eps.")
    return None


def _n4_board_from(test, mornings):
    """The board that her answers make, each in the row of the product shown."""
    moods = season_moods(test["seed"])
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for m in mornings:
        x = 0
        if moods[m["morning"]] < test["p"][m["row"]]:
            x = 1
        rows[m["row"]].append(x)
    return rows


def _n4_shape(test, answer):
    """(what, hint) when the season's answer is no board with one row per product, or None."""
    rows = len(test["p"])
    if answer is None:
        return ("your function gave no answer (None)", "end the function with return rows, after the last loop.")
    if not isinstance(answer, list):
        return ("your function gave %s, and a board is a list of rows" % _n4_short(answer),
                "return rows, the board: one list per product.")
    if len(answer) == 0:
        return ("your function gave an empty board, []",
                "the board starts with one empty row per product: rows.append([]) in the loop over the products.")
    for row in answer:
        if not isinstance(row, list):
            return ("your function gave %s, a board with numbers next to its rows" % _n4_short(answer),
                    "add each dot to the row of the product shown: rows[product].append(dot). rows.append(dot) puts it "
                    "next to the rows.")
    if len(answer) != rows:
        return ("your function gave a board of %s for %s" % (_n4_count(len(answer), "row"), _n4_count(rows, "product")),
                "one row per product: the board starts with one empty row for each probability in p.")
    for r in range(rows):
        for x in answer[r]:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return ("your function gave a board whose row %d holds %s" % (r, _n4_short(x)),
                        "each dot is her answer, 1 or 0: dot = tap(p[product], moods[morning]).")
    return None


def _n4_board_problem(test, answer, mornings):
    """(what, hint) when the board differs from the one her answers make, or None. More answers than mornings are
    left to the check of the mornings, which names them."""
    built = _n4_board_from(test, mornings)
    if answer == built or len(mornings) > test["mornings"]:
        return None
    k = len(test["p"])
    dots = 0
    same_lengths = True
    for r in range(k):
        dots = dots + len(answer[r])
        if len(answer[r]) != len(built[r]):
            same_lengths = False
    want = len(mornings)
    if dots < want:
        if dots == min(k, want) and want > k:
            return ("your board has the %d dots of the initialization and none after it" % dots,
                    "step 4, update: in the loop over the remaining mornings, add each answer at the end of the row of "
                    "the product shown, rows[product].append(dot). The next spin then sees the new estimate.")
        return ("your board has %s for %s" % (_n4_count(dots, "dot"), _n4_count(want, "morning")),
                "add each answer to the row of the product shown, once every morning: rows[product].append(dot).")
    if dots > want:
        extra = True
        for r in range(k):
            if len(answer[r]) == 0 or answer[r][1:] != built[r]:
                extra = False
        if extra:
            return ("every row of your board starts with a dot before the first morning",
                    "the board starts with one empty row per product: rows.append([]).")
        return ("your board has %s for %s" % (_n4_count(dots, "dot"), _n4_count(want, "morning")),
                "one dot per morning: append exactly once in each round of each loop.")
    if same_lengths:
        return ("every row has the right number of dots, and some dots are not her answers",
                "add her answer itself: dot = tap(p[product], moods[morning]), then rows[product].append(dot).")
    for r in range(k):
        if len(answer[r]) != len(built[r]):
            return ("the row of %s has %s, and your mirror showed it %s" % (
                _n4_product(test, r), _n4_count(len(answer[r]), "dot"), _n4_count(len(built[r]), "time")),
                "her answers went into other rows. Add each dot to the row of the product shown: "
                "rows[product].append(dot).")
    return None


def _n4_game_morning(test, t):
    """The game's morning t of a season test: its kind, the product, the numbers drawn, the next three numbers of the
    stream from that morning on, the tied rows and the board before it."""
    position = 0
    for m in range(t):
        position = position + _N4_DRAWS[test["kinds"][m]]
    numbers = strategy_numbers(test["seed"], "epsilon-greedy")
    for j in range(position):
        numbers()
    ahead = [numbers(), numbers(), numbers()]
    kind = test["kinds"][t]
    board = []
    for k in range(len(test["p"])):
        board.append([])
    for m in range(t):
        board[int(test["shown"][m])].append(int(test["dots"][m]))
    return {"kind": kind, "product": int(test["shown"][t]), "nums": ahead[:_N4_DRAWS[kind]], "ahead": ahead,
            "tied": test["ties"].get(t, []), "board": board}


def _n4_first_difference(test, mornings):
    """The first morning on which the mirror's morning, product or draws differ from the game's, or the number of
    mornings both played when they agree up to there."""
    j = 0
    while j < len(mornings) and j < test["mornings"]:
        m = mornings[j]
        if m["morning"] != j or m["row"] != int(test["shown"][j]) or len(m["draws"]) != _N4_DRAWS[test["kinds"][j]]:
            break
        j = j + 1
    return j


def _n4_morning_problem(fn, test, mornings, p_read):
    """(what, hint) for the first morning on which the mirror differs from the game's, or None."""
    n = test["mornings"]
    k = len(test["p"])
    j = _n4_first_difference(test, mornings)
    if j == len(mornings) and j == n:
        return None
    if j == len(mornings):
        if j < k:
            return ("your mirror stopped during the initialization, after %s" % _n4_count(j, "morning"),
                    "the initialization shows every product once: for morning in range(len(p)).")
        if j == k:
            return ("your mirror stopped after the %d mornings of the initialization, and the season has %d" % (k, n),
                    "steps 2 to 4 run on every remaining morning: for morning in range(len(p), len(moods)), with the "
                    "spin, the sample and the update inside that loop.")
        hint = "the remaining mornings run to the last morning of the season: for morning in range(len(p), len(moods))."
        if j == k + 1:
            hint = hint + " Put return rows after that loop, at the left edge of the function's body."
        return ("your mirror played %s of the %d" % (_n4_count(j, "morning"), n), hint)
    if j == n:
        return ("your mirror asked for her answer %s in a season of %d mornings" % (_n4_count(len(mornings), "time"), n),
                "one answer per morning: call tap once in each round of each loop.")
    m = mornings[j]
    got = m["morning"]
    if got != j:
        if len(mornings) == n:
            return ("on %s her answer came from the mood of morning %d" % (_n4_morning(j), got),
                    "each morning has its own mood: dot = tap(p[product], moods[morning]).")
        if j > 0 and got == mornings[j - 1]["morning"]:
            return ("your mirror asked for her answer on %s twice" % _n4_morning(got),
                    "one answer per morning: call tap once in each round of the loop.")
        if j == k and got < k:
            return ("after the initialization your mirror started again from morning %d" % got,
                    "the spins start on the first morning after the initialization: for morning in range(len(p), "
                    "len(moods)).")
        if j == 0:
            return ("your season started with the mood of morning %d" % got,
                    "Python counts the mornings from 0: range(len(p)) gives 0, 1, 2 and so on, and moods[0] is the mood "
                    "of the first morning.")
        if got > j and j == k:
            return ("your mirror skipped %s, the first morning after the initialization" % _n4_morning(j),
                    "the remaining mornings start right after the initialization, at morning len(p): for morning in "
                    "range(len(p), len(moods)).")
        if got > j:
            return ("your mirror skipped %s" % _n4_morning(j),
                    "range(len(p), len(moods)) gives every remaining morning, one after the other.")
        return ("on %s her answer came from the mood of morning %d" % (_n4_morning(j), got),
                "each morning has its own mood: dot = tap(p[product], moods[morning]).")
    if j < k:
        if len(m["draws"]) > 0:
            return ("on %s, during the initialization, your mirror drew %s from next_number"
                    % (_n4_morning(j), _n4_count(len(m["draws"]), "number")),
                    "spin the roulette only after the initialization: on mornings 0 to len(p) - 1 the mirror shows "
                    "product number morning, from the top row down (product = morning), and the spins start on morning "
                    "len(p), once every product has a dot.")
        return ("on %s your mirror showed %s. The initialization shows each product once, from the top row down: %s"
                % (_n4_morning(j), _n4_product(test, m["row"]), _n4_product(test, j)),
                "product = morning: morning 0 shows row 0, morning 1 row 1, and so on.")
    game = _n4_game_morning(test, j)
    best = 0
    for r in range(k):
        if test["p"][r] > test["p"][best]:
            best = r
    if p_read and m["row"] == best and m["row"] != game["product"]:
        return ("on %s your mirror showed %s, the highest probability. %s" % (
            _n4_morning(j), _n4_product(test, m["row"]), _n4_says(test, game)),
            "the mirror does not know p: only tap reads it. 🥇 is the highest estimate on its board: "
            "pick_highest(estimates(rows), next_number).")
    first = None
    if m["draws"]:
        first = m["draws"][0]
    cmps = []
    for place, other, below in m["cmps"]:
        if place is not None and first is not None:
            place = place - first
        cmps.append((place, other, below))
    what = "on %s %s. Your mirror showed %s and drew %s from next_number" % (
        _n4_morning(j), _n4_says(test, game), _n4_product(test, m["row"]), _n4_count(len(m["draws"]), "number"))
    return what, _n4_roulette_hint("epsilon_greedy", fn, test, game, m["row"], len(m["draws"]), cmps)


def _n4_season_failure(fn):
    """Task B: (head, what, hint) for the first season that fn gets wrong, "gap", or None when it gets them all."""
    eps_read = False
    for test in _N4_SEASONS:
        run = _n4_run(fn, _n4_season_inputs(test))
        mornings, p_read, read = _n4_mornings_of(run["log"])
        if read:
            eps_read = True
        expected = []
        for k in range(len(test["p"])):
            expected.append([])
        for t in range(test["mornings"]):
            expected[int(test["shown"][t])].append(int(test["dots"][t]))
        same = False
        if run["error"] is None:
            try:
                same = _n4_is_board(run["answer"]) and run["answer"] == expected
            except Exception:
                same = False
        if same and _n4_first_difference(test, mornings) != len(mornings):
            same = False
        if same and len(mornings) != test["mornings"]:
            same = False
        if same:
            continue
        head = "%s, %s" % (test["call"], test["about"])
        if run["answer"] is Ellipsis or "ellipsis" in str(run["error"]).lower():
            return "gap"
        if run["error"] is not None:
            # A morning that went wrong before the error is where to look first
            if _n4_first_difference(test, mornings) < len(mornings):
                problem = _n4_morning_problem(fn, test, mornings, p_read)
                return (head, problem[0], problem[1])
            where = ""
            if run["line"]:
                where = " on the line  %s" % run["line"]
            return (head, "your function stopped with an error%s. %s" % (where, run["error"]),
                    _n4_explain_error("epsilon_greedy", run, len(test["p"])))
        problem = _n4_shape(test, run["answer"])
        if problem is None:
            for m in mornings:
                if m["swapped"] and problem is None:
                    problem = ("on %s her answer is the wrong way round: she taps when her mood is below the "
                               "probability" % _n4_morning(m["morning"]),
                               "tap takes the probability first, then the mood: dot = tap(p[product], moods[morning]).")
        if problem is None:
            for m in mornings:
                if m["row"] is None and problem is None:
                    if isinstance(m["value"], int) and not isinstance(m["value"], bool):
                        problem = ("on %s her answer came from %d, a row number in the place of a probability"
                                   % (_n4_morning(m["morning"]), m["value"]),
                                   "tap takes the probability of the product shown, p[product], then the mood: "
                                   "dot = tap(p[product], moods[morning]).")
                    else:
                        problem = ("on %s her answer came from a probability that is not one of p"
                                   % _n4_morning(m["morning"]),
                                   "use p, the probabilities that the function gets: dot = tap(p[product], "
                                   "moods[morning]). The check also tries the other customer and a collection.")
        if problem is None:
            problem = _n4_board_problem(test, run["answer"], mornings)
        if problem is None:
            problem = _n4_morning_problem(fn, test, mornings, p_read)
        if problem is None:
            problem = ("your board differs from the game's",
                       "compare your code with the four steps above the cell, then open the hints.")
        return (head, problem[0], problem[1])
    if not eps_read:
        hint = "compare the spin with the eps that the function gets: if spin < eps."
        if _n4_uses(fn, "spin_the_roulette"):
            hint = "hand the roulette the eps that epsilon_greedy gets: spin_the_roulette(rows, next_number, eps)."
        return ("epsilon_greedy on the %d seasons of its check" % len(_N4_SEASONS),
                "your mirror plays the game's seasons, and the eps that it gets never reaches the roulette", hint)
    return None


def _n4_key(name):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    key = [getattr(globals().get(name), "__code__", None)]
    for other in _N4_NEEDS[name]:
        key.append(getattr(globals().get(other), "__code__", None))
    return tuple(key)


def _n4_failure(name, fn):
    if name == "spin_the_roulette":
        return _n4_roulette_failure(fn)
    return _n4_season_failure(fn)


def _n4_works(name):
    """True when the function called name exists, is finished and passes its tests. Remembered for each version of
    it, so play can ask on every season."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n4_has_gap(fn):
        return False
    for other in _N4_NEEDS[name]:
        if _n4_uses(fn, other) and not _n4_works(other):
            return False
    key = _n4_key(name)
    if key not in _N4_WORKING:
        _N4_WORKING[key] = _n4_failure(name, fn) is None
    return _N4_WORKING[key]


def _n4_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
                    "cell, then run this check again." % name)
    if _n4_has_gap(fn):
        print(not_finished)
        return
    for other in _N4_NEEDS[name]:
        if _n4_uses(fn, other) and not _n4_works(other):
            needed = globals().get(other)
            if callable(needed) and _n4_has_gap(needed):
                print("⏳ %s uses your %s from %s, which is not finished yet. Finish it and get check_%s(%s) to ✅ "
                      "first, then run this check again." % (name, other, _N4_TASKS[other], other, other))
                return
            print("⏳ %s uses your %s from %s, which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                  "check again." % (name, other, _N4_TASKS[other], other, other))
            return
    failure = _n4_failure(name, fn)
    if failure is None:
        print("✅ " + _N4_SUCCESS[name])
        return
    if failure == "gap":
        print(not_finished)
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint)


def check_spin_the_roulette(fn):
    """Task A: spins the roulette on seven mornings of the game (both customers, 🥇, 🎩 and a tie) and compares the
    product and the numbers drawn with the game's, number for number."""
    try:
        _n4_check("spin_the_roulette", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the three folded cells at the top again, then this cell." % error)


def check_epsilon_greedy(fn):
    """Task B: plays epsilon_greedy on seven seasons (both customers, a collection, a short season) with the game's
    stream of each season, and compares each board with the game's, morning by morning."""
    try:
        _n4_check("epsilon_greedy", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the three folded cells at the top again, then this cell." % error)


def _n4_wait(mirror, cell):
    """One calm line and True when the mirror is not ready to play, else False."""
    if not callable(mirror):
        print("⏳ %s needs your function itself, as in play(epsilon_greedy, cinderella, moods, "
              "strategy_numbers(RACE, \"epsilon-greedy\"), eps)." % cell)
        return True
    name = getattr(mirror, "__name__", "your mirror")
    if _n4_has_gap(mirror):
        task = ""
        if name in _N4_TASKS:
            task = " (%s)" % _N4_TASKS[name]
        print("⏳ %s is not finished yet%s, so there is no season to show. Finish it, run its cell, then run this cell "
              "again." % (name, task))
        return True
    for other in _N4_TASKS:
        fn = globals().get(other)
        if other == name or not _n4_uses(mirror, other):
            continue
        if callable(fn) and _n4_has_gap(fn):
            print("⏳ %s uses your %s (%s), which is not finished yet. Finish it, run its cell, then run this cell "
                  "again." % (name, other, _N4_TASKS[other]))
            return True
        if not _n4_works(other):
            print("⏳ %s uses your %s (%s), which does not work yet. Get check_%s(%s) to ✅ first, then run this cell "
                  "again." % (name, other, _N4_TASKS[other], other, other))
            return True
    return False


def play(mirror, *inputs):
    """Plays one season with your mirror and gives its board: play(epsilon_greedy, cinderella, moods, numbers, eps)
    is epsilon_greedy(cinderella, moods, numbers, eps). When the mirror is not finished yet, stops with an error or
    gives something that is not a board, play prints one calm line and gives None, so the cells below still run."""
    import contextlib, io
    try:
        if _n4_wait(mirror, "play"):
            return None
        name = getattr(mirror, "__name__", "your mirror")
        copies = []
        for x in inputs:
            if isinstance(x, list):
                copies.append(list(x))
            else:
                copies.append(x)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                answer = mirror(*copies)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check above says more." % (name, type(error).__name__,
                                                                                     error))
            return None
        rows = None
        if len(copies) > 0 and isinstance(copies[0], list):
            rows = len(copies[0])
        if not _n4_is_board(answer, rows):
            print("❌ %s gave %s, which is not a board. Its check above says more." % (name, _n4_short(answer)))
            return None
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the three folded cells at the top again, then this cell." % error)
        return None


def show_mornings(mirror, p, moods, next_number, eps, mornings):
    """Plays one season with your mirror and prints its first mornings, one line each, counted from 1 as in the
    game: the roulette (where the needle stopped, 🥇 or 🎩), the product shown and her answer. It waits, with one
    calm line, until the mirror passes its check."""
    try:
        if _n4_wait(mirror, "show_mornings"):
            return
        name = getattr(mirror, "__name__", "your mirror")
        if name in _N4_TASKS and not _n4_works(name):
            print("⏳ show_mornings waits until %s works: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return
        inputs = [[], [], _n4_stream(next_number, 0), eps]
        for k in range(len(p)):
            x = _n4_Prob(p[k])
            x.row = k
            inputs[0].append(x)
        for t in range(len(moods)):
            x = _n4_Mood(moods[t])
            x.morning = t
            inputs[1].append(x)
        run = _n4_run(mirror, inputs)
        if run["error"] is not None:
            print("❌ %s stopped with an error, %s. Its check above says more." % (name, run["error"]))
            return
        values = {}
        for kind, a, b, c, d in run["log"]:
            if kind == "draw":
                values[a] = d
        found, p_read, eps_read = _n4_mornings_of(run["log"])
        for t in range(min(mornings, len(found))):
            m = found[t]
            row = m["row"]
            product = "row %s" % row
            if isinstance(row, int) and len(p) == len(NAMES):
                product = EMOJI[row] + " " + NAMES[row].lower()
            answer = "⚪ no tap"
            if moods[m["morning"]] < p[row]:
                answer = "🟠 tap"
            if len(m["draws"]) == 0:
                roulette = "initialize"
            else:
                spin = values[m["draws"][0]]
                if spin < eps:
                    roulette = "spin %s, 🎩 a card from the hat" % _n4_spin(spin, eps)
                elif len(m["draws"]) > 1:
                    roulette = "spin %s, 🥇 a tie for the highest estimate, broken at random" % _n4_spin(spin, eps)
                else:
                    roulette = "spin %s, 🥇 the highest estimate" % _n4_spin(spin, eps)
            print("morning %d: %s, %s: %s" % (m["morning"] + 1, roulette, product, answer))
    except Exception as error:
        print("❌ show_mornings could not run (%s). Run the three folded cells at the top again, then this cell."
              % error)


print("✅ The checks of notebook 4 are ready.")

In [ ]:
#@title From notebooks 2 and 3: the Queen's mirror, Random and estimate then exploit (run me too) { display-mode: "form" }
# The mirrors of notebooks 2 and 3, as in their solutions. Run this cell; open it if you like.

def queen_mirror(p, moods):
    # The Queen's mirror knows every probability: every morning it shows the product with the highest one.
    best = 0
    for i in range(len(p)):
        if p[i] > p[best]:      # product i beats the best so far
            best = i
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        dot = tap(p[best], moods[morning])    # 1 if she taps the best product this morning, else 0
        rows[best].append(dot)                # one more dot in its row
    return rows


def random_mirror(p, moods, next_number):
    # Random draws a card from the hat each morning: every product has the same chance.
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        card = int(next_number() * len(p))    # a card from the hat: a row from 0 to len(p) - 1
        dot = tap(p[card], moods[morning])    # her answer this morning: 1 or 0
        rows[card].append(dot)                # one more dot in the card's row
    return rows


def round_robin(p, moods, n):
    # The test of estimate then exploit: n rounds, every product once per round, from the top row down.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product
    test = n * len(p)                           # n rounds of len(p) mornings
    for morning in range(test):
        product = morning % len(p)              # in turn: 0, 1, 2, 3, 4, then 0 again
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row
    return rows


def estimate_then_exploit(p, moods, next_number, n):
    # The test, then the highest estimate on every remaining morning.
    rows = round_robin(p, moods, n)             # the board after the test
    test = n * len(p)                           # the mornings of the test
    winner = pick_highest(estimates(rows), next_number)    # the highest estimate, picked once
    for morning in range(test, len(moods)):
        dot = tap(p[winner], moods[morning])    # her answer to the winner: 1 or 0
        rows[winner].append(dot)                # one more dot in the winner's row
    return rows


print("✅ queen_mirror, random_mirror and estimate_then_exploit from notebooks 2 and 3 are ready.")

## The four steps

The game told epsilon-greedy in four steps:

1. **Initialize**: at the start of the season, show each product once, from the top row down.
2. **Spin the roulette**: 🥇 the product with the highest estimate, or 🎩 a card from the hat.
3. **Sample**: show that product to the customer: tap 🟠 or no tap ⚪.
4. **Update the estimate**: add the dot to its row and compute its estimate again, p̂ = orange dots / dots.

Then steps 2 to 4 again, every morning, until the season ends. Here are the two customers, `eps`, the size of the
small slice, and the moods of the race season, the season of every comparison in the game.

In [ ]:
# The probability that each customer taps each product, in board order (invented for the game).
# Only tap reads them: the mirror itself sees only the dots of its board.
cinderella = [0.6, 0.2, 0.2, 0.4, 0.2]
rapunzel = [0.4, 0.8, 0.2, 0.2, 0.2]

# The size of the small slice of the roulette, 🎩: one spin in five, as in the game
eps = 0.2

# The race season of the game: her mood on each of its mornings
moods = season_moods(RACE)

for i in range(len(NAMES)):
    print("row", i, "   Cinderella", cinderella[i], "   Rapunzel", rapunzel[i], "   ", EMOJI[i], NAMES[i])

## The roulette in Python

The needle stops at a number between 0 and 1: `spin = next_number()`. The small slice, 🎩, holds the numbers below
`eps`, one spin in five; the large slice, 🥇, holds the rest. `strategy_numbers(RACE, "epsilon-greedy")` is the mirror's own stream of numbers
in the race season, as in the game. Each morning after the initialization the mirror draws one number for the spin,
and one more only when it needs it: for the card, when the needle stops on 🎩, and inside `pick_highest`, for a tie.

In [ ]:
# The mirror's own stream of numbers in the race season: its first 3 spins, and the card of the last one
next_number = strategy_numbers(RACE, "epsilon-greedy")
for turn in range(3):
    spin = next_number()                      # where the needle stops
    if spin < eps:
        print(spin, "-> 🎩 the small slice: a card from the hat")
    else:
        print(spin, "-> 🥇 the large slice: the highest estimate")

card = int(next_number() * 5)                 # the next number gives the card: a row from 0 to 4
print("The card:", card, EMOJI[card], NAMES[card])

**Reading the result.** These are the spins of mornings 6, 7 and 8 of the game's walkthrough, counted from 1 as
in the game. On mornings 6 and 7 the needle stops at 0.32 and 0.6, on 🥇: the memory-foam mattress, the highest estimate after its tap on
morning 4. On morning 8 it stops at 0.07, below 0.2, on 🎩, and the next number gives the card: row 4, the satnav. The
mirror draws a card only after a spin on 🎩. Drawn in this order, the same stream makes the same choices as in the
game.

## 🎯 Task A. Spin the roulette

Write `spin_the_roulette(rows, next_number, eps)`: step 2 of one morning after the initialization. It gets the board
so far, `rows`, the mirror's stream `next_number` and `eps`, and returns the row of the product to show:

1. **Spin**: `spin = next_number()`, where the needle stops.
2. **🎩 the small slice**: when `spin < eps`, a card from the hat, `int(next_number() * len(rows))`: a row from 0
   to 4, since `len(rows)` is 5.
3. **🥇 the large slice**: otherwise the row with the highest estimate on the board,
   `pick_highest(estimates(rows), next_number)`, as in notebook 3. It draws a number only for a tie.

The roulette sees only the board: `p` is not among its inputs. Draw the numbers in this order, the spin first and
the card only on 🎩, and your roulette makes the game's choices, number for number.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Step 1: next_number() gives the next number of the stream, with brackets, as in the cell of the roulette.
# Step 2: the needle stops on 🎩 when the spin is below eps: if spin < eps. The card is a new number, drawn
# after the spin: int(next_number() * len(rows)).
# Step 3: estimates(rows) gives the estimate of every row, and pick_highest(values, next_number) the row with
# the highest one.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     spin = next_number()
#     if spin < eps:
#         product = int(next_number() * len(rows))
#     else:
#         product = pick_highest(estimates(rows), next_number)

In [ ]:
# ✅ SOLUTION, task A: spin the roulette
def spin_the_roulette(rows, next_number, eps):
    # rows: the board so far, one list of 1s and 0s per product. The roulette sees only the board.
    # next_number: the mirror's own stream of numbers. eps: the size of the small slice, 🎩.
    # Returns the row of the product to show this morning.
    spin = next_number()                        # where the needle stops: a number between 0 and 1
    if spin < eps:                              # 🎩 the small slice
        product = int(next_number() * len(rows))                # a card from the hat: any row at random
    else:                                       # 🥇 the large slice
        product = pick_highest(estimates(rows), next_number)    # the highest estimate, a tie at random
    return product

In [ ]:
check_spin_the_roulette(spin_the_roulette)

## 🎯 Task B. Epsilon-greedy

Write `epsilon_greedy(p, moods, next_number, eps)`. It plays a whole season and returns its board, in the four
steps. The board is started for you, and so is each loop:

1. **Initialize**: on mornings 0 to 4 (Python counts the mornings from 0), each product once, from the top row down: on
   morning `m`, the product of row `m`.
2. **Spin the roulette**: your `spin_the_roulette` from task A gives the product.
3. **Sample**: her answer to it, `tap(p[product], moods[morning])`: 1 for a tap, 0 for no tap.
4. **Update the estimate**: add the dot at the end of the product's row. `estimates(rows)` computes every estimate from the board,
   so the next spin sees the new one.

Steps 2 to 4 run on every morning after the initialization: `range(len(p), len(moods))` gives 5, 6, 7 and so on
up to 99. With `eps` and the stream `strategy_numbers(RACE, "epsilon-greedy")`, your mirror plays the game's race season,
morning by morning.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Step 1: on morning m of the initialization the mirror shows row m, so the product is the morning itself.
# Step 2: spin_the_roulette(rows, next_number, eps) gives the product, from the board so far.
# Step 3: her answer is tap(p[product], moods[morning]), as in the initialization.
# Step 4: rows[product].append(dot) adds the dot at the end of the product's row.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for morning in range(len(p)):
#         product = morning
#
#     for morning in range(len(p), len(moods)):
#         product = spin_the_roulette(rows, next_number, eps)
#         dot = tap(p[product], moods[morning])
#         rows[product].append(dot)

In [ ]:
# ✅ SOLUTION, task B: epsilon-greedy
def epsilon_greedy(p, moods, next_number, eps):
    # p: the probability of a tap for each product, in board order. Only tap reads it.
    # moods: her mood on each morning. next_number: the mirror's own stream. eps: the small slice, 🎩.
    # Returns the board of the whole season.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product

    # 1. Initialize: each product once, from the top row down
    for morning in range(len(p)):
        product = morning                       # morning 0 shows row 0, morning 1 row 1, and so on
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row

    # Steps 2 to 4 on every morning after the initialization
    for morning in range(len(p), len(moods)):
        product = spin_the_roulette(rows, next_number, eps)    # 2. spin: 🥇 or 🎩
        dot = tap(p[product], moods[morning])   # 3. sample: her answer, 1 or 0
        rows[product].append(dot)               # 4. update: the next spin sees the new estimate
    return rows

In [ ]:
check_epsilon_greedy(epsilon_greedy)

## Cinderella's season, ε = 0.2

💭 **Think first.** After the initialization the mirror spins 95 times, and one spin in five lands on 🎩. About how many cards from
the hat do you expect in the season? After the initialization only the memory-foam mattress has a tap, and the robot vacuum, Cinderella's favourite,
has none. How can the mirror ever find the vacuum?

`play(epsilon_greedy, cinderella, moods, strategy_numbers(RACE, "epsilon-greedy"), eps)` runs your mirror on the race season
and gives its board. While the function is not finished, it prints one calm line and gives `None`.

In [ ]:
# Epsilon-greedy at Cinderella's on the race season, with the game's eps
eg_rows = play(epsilon_greedy, cinderella, moods, strategy_numbers(RACE, "epsilon-greedy"), eps)
show_board(eg_rows)
print("The rate of epsilon-greedy:", rate(eg_rows))

`show_mornings` plays the same season again and prints its first 25 mornings, the way the game's walkthrough told
them: the roulette, the product and her answer.

In [ ]:
# The first 25 mornings of the same season, one line each, counted from 1 as in the game
show_mornings(epsilon_greedy, cinderella, moods, strategy_numbers(RACE, "epsilon-greedy"), eps, 25)

**Reading the result.** In the first 25 mornings the needle stops on 🎩 on mornings 8, 18 and 23, counted from 1 as in the
game. On morning 8 the card is the satnav. On morning 18 the hat finds the vacuum, and Cinderella taps: its estimate becomes 1/2 = 0.5,
above the mattress's 4/12 = 0.33, so from morning 19 on 🥇 picks the vacuum. Without the hat, 🥇 would have shown the mattress on every
morning of the season: the other estimates were 0, and an estimate changes only when its product is shown.

In the whole season the needle stops on 🎩 16 times in 95 spins, on mornings 8, 18, 23, 45, 50, 51, 53, 54, 55, 63, 83, 84, 85, 87, 93 and 97. One spin in five makes about 19. The board
ends with 100 dots: 71 on the vacuum, 6 on the ladder, 2 on the blower, 16 on the mattress and 5 on the satnav. The board shows the first 30 dots of the vacuum's row and counts the other 41. **The rate of
epsilon-greedy: 47 taps in 100 mornings, 0.47.** These are the game's numbers.

## Against the Queen's mirror

In the game, epsilon-greedy ran against the Queen's mirror and the earlier mirrors on the same season. Here are
the four rates on the race season, at Cinderella's: the Queen's mirror, Random from notebook 2, estimate then exploit
with n = 5 from notebook 3, and your epsilon-greedy. The first three come from the third folded cell at the top.

In [ ]:
# The four mirrors on the race season: the same moods, and each mirror its own stream
queen_rows = play(queen_mirror, cinderella, moods)
random_rows = play(random_mirror, cinderella, moods, strategy_numbers(RACE, "random"))
ete_rows = play(estimate_then_exploit, cinderella, moods, strategy_numbers(RACE, "estimate then exploit"), 5)
print("The Queen's mirror:", rate(queen_rows))
print("Random:", rate(random_rows))
print("Estimate then exploit, n = 5:", rate(ete_rows))
print("Epsilon-greedy:", rate(eg_rows))

**Reading the result.** The Queen's mirror 0.6, Random 0.31, estimate then exploit 0.46 and epsilon-greedy 0.47: 60, 31,
46 and 47 taps in the same 100 mornings. One season is one roll of luck: below, the four mirrors play many seasons.

💭 **Think first.** Over many seasons the Queen's mirror gets about 0.6, the vacuum's probability, and Random about 0.32,
the average of the five probabilities. Where do you expect epsilon-greedy, and estimate then exploit?

`average_rates(p, eps, first_seed, seasons)` plays the four mirrors for many seasons, one season per seed, and
prints the rate of all those mornings. In each season every mirror meets the same moods. The cell takes a few
seconds.

In [ ]:
# Given: the taps of a board, and the four mirrors over many seasons
def count_taps(rows):
    # All the 1s on the board, added up
    taps = 0
    for row in rows:
        for dot in row:
            taps = taps + dot
    return taps


def average_rates(p, eps, first_seed, seasons):
    # Plays the Queen's mirror, Random, estimate then exploit (n = 5) and epsilon-greedy for many seasons, one
    # season per seed, and prints the rate of all those mornings. In each season every mirror meets the same moods.
    queen_taps = 0
    random_taps = 0
    ete_taps = 0
    eg_taps = 0
    for season in range(seasons):
        seed = first_seed + season
        moods = season_moods(seed)
        eg_rows = play(epsilon_greedy, p, moods, strategy_numbers(seed, "epsilon-greedy"), eps)
        if eg_rows == None:
            return                  # epsilon_greedy is not ready yet: play has said why
        eg_taps = eg_taps + count_taps(eg_rows)
        queen_taps = queen_taps + count_taps(play(queen_mirror, p, moods))
        random_taps = random_taps + count_taps(play(random_mirror, p, moods, strategy_numbers(seed, "random")))
        ete_rows = play(estimate_then_exploit, p, moods, strategy_numbers(seed, "estimate then exploit"), 5)
        ete_taps = ete_taps + count_taps(ete_rows)
    mornings = seasons * SEASON
    print("Seasons:", seasons, "   first seed:", first_seed, "   mornings in all:", mornings)
    print("The Queen's mirror:", queen_taps / mornings)
    print("Random:", random_taps / mornings)
    print("Estimate then exploit, n = 5:", ete_taps / mornings)
    print("Epsilon-greedy:", eg_taps / mornings)

In [ ]:
# 1,000 seasons at Cinderella's: seeds 1 to 1000
average_rates(cinderella, eps, 1, 1000)

**Reading the result.** Over these 1,000 seasons: the Queen's mirror 0.6, epsilon-greedy 0.47, estimate then exploit
0.46, Random 0.32. Epsilon-greedy comes closest to the Queen's mirror. It pays for exploring all season long: it first
has to find the vacuum, and even then one spin in five goes to the hat, which shows another product four times in five. Estimate then
exploit pays at the start, with 25 mornings of test, and keeps its pick even when the test chose wrong.

## Back to the game

That is notebook 4. Section 5 of the game, **UCB**, shows a mirror that explores the products it knows least. Notebook 5 comes after it.